In [81]:
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch
import torch.nn as nn 
from torch.nn import functional as F


#### setting-up data

In [82]:
file_path = "/Users/millan/Desktop/September Coding/dostoyevsky.txt"
with open(file_path, 'r') as f:
    text = f.read()


In [83]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

In [84]:
characters = sorted(list(set(text)))
print(''.join(characters))



 !"&'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyzÀÆÉàâäæçèéêëîïôöüŒœ—‘’“”


#### Parameters

In [85]:
vocab_size = len(characters)
batch_size = 64
context_length = 256
learning_rate = 3e-4
evaluation_iterations = 200
eval_interval = 500
max_iterations = 10_000
embd_dim = 384
head_size = 64
n_head = 6
n_layer = 6
dropout = 0.2

#### encoder + decoder

In [86]:
int_to_str = {i : ch for i, ch in enumerate(characters)}
str_to_int = {ch : i for i, ch in enumerate(characters)}

def encode(s):
    return [str_to_int[c] for c in s]

def decode(l):
    return ''.join([int_to_str[i] for i in l])

In [87]:
print(encode('hello'))
print(decode([61, 58, 65, 65, 68]))

[61, 58, 65, 65, 68]
hello


#### Tokeniser

In [88]:
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)

torch.Size([7063175]) torch.int64


In [89]:
#train/test split for tokenised data

n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [90]:
x = train_data[:context_length]
y = train_data[1:context_length + 1]

for t in range(context_length):
    context = x[:t + 1]
    target = y[t]
    print(f'for input {context} target is {target}')

for input tensor([40]) target is 25
for input tensor([40, 25]) target is 42
for input tensor([40, 25, 42]) target is 44
for input tensor([40, 25, 42, 44]) target is 1
for input tensor([40, 25, 42, 44,  1]) target is 33
for input tensor([40, 25, 42, 44,  1, 33]) target is 0
for input tensor([40, 25, 42, 44,  1, 33,  0]) target is 0
for input tensor([40, 25, 42, 44,  1, 33,  0,  0]) target is 0
for input tensor([40, 25, 42, 44,  1, 33,  0,  0,  0]) target is 27
for input tensor([40, 25, 42, 44,  1, 33,  0,  0,  0, 27]) target is 32
for input tensor([40, 25, 42, 44,  1, 33,  0,  0,  0, 27, 32]) target is 25
for input tensor([40, 25, 42, 44,  1, 33,  0,  0,  0, 27, 32, 25]) target is 40
for input tensor([40, 25, 42, 44,  1, 33,  0,  0,  0, 27, 32, 25, 40]) target is 44
for input tensor([40, 25, 42, 44,  1, 33,  0,  0,  0, 27, 32, 25, 40, 44]) target is 29
for input tensor([40, 25, 42, 44,  1, 33,  0,  0,  0, 27, 32, 25, 40, 44, 29]) target is 42
for input tensor([40, 25, 42, 44,  1, 33,  0

#### batches

In [91]:
torch.manual_seed(3333)

batch_size = 4
context_length = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    index = torch.randint(len(data) - (context_length + 1), (batch_size,))
    x = torch.stack([data[i : i + context_length] for i in index])
    y = torch.stack([data[i + 1 : i + context_length + 1] for i in index])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')

### actual model

In [92]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embd_dim, head_size, bias = False )
        self.query = nn.Linear(embd_dim, head_size, bias = False )
        self.value = nn.Linear(embd_dim, head_size, bias = False )
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        weights = q @ k.transpose(-2,-1) * k.shape[-1] ** -0.5
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weights = F.softmax(weights, dim=-1)
        weights = self.dropout(weights)

        v = self.value(x)
        out = weights @ v

        return out



In [93]:
class MultiHead(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for i in range(num_heads)])
        self.projection = nn.Linear(embd_dim, embd_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.projection(out)
        return out


In [94]:
class FeedForward(nn.Module):

    def __init__(self, embd_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embd_dim, 4 * embd_dim), 
            nn.ReLU(), 
            nn.Linear(4 * embd_dim, embd_dim),
            nn.Dropout(dropout),)
    
    def forward(self, x):
        return self.net(x)
        

In [95]:
class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        current_mean = x.mean(-1, keepdim=True)
        current_std = x.std(-1, keepdim=True)
        x_pre_act = (x - current_mean) / (current_std + self.eps)
        self.out = self.gamma * x_pre_act + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

In [96]:
class Block(nn.Module):

    def __init__(self, embd_dim, n_head):
        super().__init__()
        head_size = embd_dim // n_head
        self.sa = MultiHead(n_head, head_size)
        self.ffwd = FeedForward(embd_dim)

        self.ln1 = nn.LayerNorm(embd_dim)
        self.ln2 = nn.LayerNorm(embd_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
    


In [97]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(3333)


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, embd_dim)
        self.position_embedding_table = nn.Embedding(context_length, embd_dim)

        self.blocks = nn.Sequential(*[Block(embd_dim, n_head = n_head) for i in range(n_layer)])
        self.ln_f = nn.LayerNorm(embd_dim) #last layer norm
        self.lm_head = nn.Linear(embd_dim, vocab_size)
        

    def forward(self, index, targets=None):

        B, T = index.shape

        token_embeddings = self.token_embedding_table(index)
        position_embeddings = self.position_embedding_table(torch.arange(T, device=device))
        x = token_embeddings + position_embeddings
        x = self.blocks(x)
        logits = self.lm_head(x)

        if targets == None:
            loss = None
        else:
        
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_token_size):
        for i in range(max_token_size):
            index_cond = index[:, -context_length:]
            logits, loss = self(index_cond) #inside generate self is the model isntance (so getting the logits)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat(( index, index_next), dim=1)

        return index




model = BigramLanguageModel()
m = model.to(device)
logits, loss = m(xb, yb)
print(loss)

index = torch.zeros((1,1), dtype=torch.long, device=device)
decode(m.generate(index, 100)[0].tolist())

tensor(4.9574, device='mps:0', grad_fn=<NllLossBackward0>)


'\nZ,\n è7"AÆcaŒz3z)æÆ_a\n5iw\'3"?5wqiÆO;F;2ä\'ÀOl1xQ?pSFö1âmé_“a2VäX0-q_ö6QNFa:KVfÆA3VGL—[kzDkäæ3l2OW:Z“qQ'

In [98]:
# evaluation_iterations = 200, is how many batches sample to estiamte true mean

@torch.no_grad
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(evaluation_iterations)
        for k in range(evaluation_iterations):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    return out

In [99]:
optimiser = torch.optim.AdamW(m.parameters(), lr=learning_rate)
    


for iteration in range(max_iterations):
    if iteration % eval_interval == 0:
        losses = estimate_loss()
        print(f"at step{iteration}, train loss is {losses['train']}, val loss is {losses['val']} ")



    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimiser.zero_grad(set_to_none=True)
    loss.backward()
    optimiser.step()





at step0, train loss is 4.961701393127441, val loss is 4.969149589538574 
at step500, train loss is 2.5146477222442627, val loss is 2.5711469650268555 
at step1000, train loss is 2.349510431289673, val loss is 2.408426523208618 
at step1500, train loss is 2.292482614517212, val loss is 2.386334180831909 
at step2000, train loss is 2.201958179473877, val loss is 2.291837692260742 
at step2500, train loss is 2.1859703063964844, val loss is 2.231727361679077 
at step3000, train loss is 2.180940628051758, val loss is 2.250302314758301 
at step3500, train loss is 2.172705888748169, val loss is 2.1812758445739746 
at step4000, train loss is 2.065652370452881, val loss is 2.149944305419922 
at step4500, train loss is 2.137064218521118, val loss is 2.1897079944610596 
at step5000, train loss is 2.123561143875122, val loss is 2.143792152404785 
at step5500, train loss is 2.0904853343963623, val loss is 2.1316068172454834 
at step6000, train loss is 2.0602900981903076, val loss is 2.146236658096

#### generate from it

In [101]:
index = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(index, 1_000)[0].tolist()))


me as him ”

Sse tood calle see don't escance And you doarted a can iwo
seay it va!
so go----hom, upy the soon and drawer my reten-the pud pos he gank-no, and Nor jot ther use.”

“You
ma!

"No, know, that
the,” the I.” he tud I very o-coved me dreat now.”

“Ans hereroure in
maken, sh wite
again, of evaing so on
be
everaced.." And pour to throv’s jones but h. Throafuch to if you doane!ted; one tell?... Nnot! You don’t my gives eve is not! ‘-----do a sred evereot  to time other and in tlion know you good sviced to me!...
kovve did you know mad you
all I so a teading brience.”

“You did scend
not ot saye sees tood
won’t pole the a cereupted upon it.
Lorkaved you stand to the hearts for see was coitch abspe to alto man’t no

“I had a spice laik of you dreading, ‘I don do you damay ow j’s of it s dicwery every mand  to  Not about by quare. Why jayd I gome wind hose...... I say. Trakoved is peoen greld your sohel.? He keage dooked his over heat treare, for I

" He at deel no greess you feot

In [102]:
index = torch.zeros((1,1), dtype=torch.long, device=device)
generated_text = decode(m.generate(index, 2_000)[0].tolist())

with open('generated_output.txt', 'w') as f:
    f.write(generated_text)